# Week 4 — SAHI Sliced Inference + Domain Shift Analysis

**Two goals this week:**
1. Apply SAHI to compare standard vs sliced inference on aerial dataset
2. Test the fine-tuned model on unseen internet images — document domain shift behavior

**Why SAHI works:** Standard detection downsamples your image to 640×640. A 10px object in a 4K image becomes ~2px — invisible. SAHI slices the image into overlapping 640×640 tiles, runs detection on each tile at full resolution, then merges results with NMS. The object is no longer tiny relative to the tile.

**Why domain shift matters:** A model with 92% mAP on training data can fail completely on slightly different real-world images. Understanding when and why is critical for production deployment.

In [ ]:
!pip install sahi ultralytics python-dotenv -q

In [ ]:
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
from dotenv import load_dotenv

from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
from ultralytics import YOLO

load_dotenv()

# ── Update these paths for your environment ───────────────────────────────────
MODEL_PATH = "../runs/detect/small_object_run1/weights/best.pt"  # fine-tuned model
DATASET_PATH = "./drone-aerial-1"                                 # downloaded dataset

# Windows absolute path example (uncomment and update if needed):
# MODEL_PATH = r"E:\path\to\runs\detect\small_object_run1\weights\best.pt"
# DATASET_PATH = r"E:\path\to\notebooks\drone-aerial-1"

print(f"Model path: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {os.path.exists(DATASET_PATH)}")

## Step 1 — Load models for standard and SAHI inference

In [ ]:
# Standard YOLOv8 model — fine-tuned
yolo_model = YOLO(MODEL_PATH)

# SAHI wrapper — same fine-tuned model
sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=0.3,
    device="cpu",  # change to "cuda" if GPU available
)

print("Both models loaded successfully.")

## Step 2 — Pick a test image from the aerial dataset

In [ ]:
# Find test images from dataset
test_images = glob.glob(os.path.join(DATASET_PATH, "test", "images", "*.jpg"))
print(f"Found {len(test_images)} test images")

# Pick one with visible objects — try different indices
TEST_IMAGE = test_images[0]
print(f"Using: {TEST_IMAGE}")

img = Image.open(TEST_IMAGE)
print(f"Image size: {img.size}")

## Step 3 — Standard inference (no SAHI)

In [ ]:
# Run standard YOLOv8 inference
standard_results = yolo_model(TEST_IMAGE, conf=0.3)
standard_count = len(standard_results[0].boxes)

print(f"Standard inference: {standard_count} detections")

# Save result image
standard_img = standard_results[0].plot()
Image.fromarray(standard_img[:, :, ::-1]).save("results/standard_result.png")
print("Saved: results/standard_result.png")

## Step 4 — SAHI sliced inference

In [ ]:
# Run SAHI sliced inference
# slice_height/width: tile size — smaller = more tiles = better for tiny objects
# overlap_ratio: how much tiles overlap — 0.2 = 20% overlap prevents missing objects at tile boundaries
sahi_result = get_sliced_prediction(
    TEST_IMAGE,
    sahi_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)

sahi_count = len(sahi_result.object_prediction_list)
print(f"SAHI inference: {sahi_count} detections")

# Save SAHI result
sahi_result.export_visuals(export_dir="results/", file_name="sahi_result")
print("Saved: results/sahi_result.png")

## Step 5 — Side-by-side comparison

In [ ]:
import cv2

standard_img = standard_results[0].plot()
standard_img_rgb = standard_img[:, :, ::-1]

sahi_img = np.array(Image.open("results/sahi_result.png"))

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(standard_img_rgb)
axes[0].set_title(f"Standard YOLOv8\n{standard_count} detections", fontsize=14)
axes[0].axis('off')

axes[1].imshow(sahi_img)
axes[1].set_title(f"SAHI Sliced Inference\n{sahi_count} detections", fontsize=14)
axes[1].axis('off')

plt.suptitle("Before vs After SAHI", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig("results/before_after_sahi.png", dpi=150, bbox_inches='tight')
plt.show()

diff = sahi_count - standard_count
pct = (diff / max(standard_count, 1)) * 100
print(f"\nResult: SAHI found {diff} more objects ({pct:+.0f}%)")

if diff == 0:
    print("\nNote: 0% improvement is expected when using a fine-tuned model at 92% mAP50.")
    print("SAHI shows larger gains on weaker/generic models or lower-quality images.")
    print("Fine-tuning and SAHI are complementary — fine-tuning specializes the model,")
    print("SAHI compensates for resolution loss when you cannot retrain.")

## Results Summary

| Method | Detections | Notes |
|--------|-----------|-------|
| Standard YOLOv8 | 3 | baseline |
| SAHI (slice 640, overlap 0.2) | 3 | fine-tuned model already at 92% mAP — SAHI improvement marginal; shows fine-tuning can replace inference-time augmentation |

## The interview story

> "I fine-tuned YOLOv8 on a small object dataset. Standard detection missed many tiny objects because they become ~2 pixels after downsampling. I applied SAHI — it slices the image into overlapping 640×640 tiles, runs detection on each tile at full resolution, then merges results with NMS. On a strong fine-tuned model (92% mAP50), SAHI improvement was marginal — demonstrating that domain-specific fine-tuning can be as powerful as inference-time augmentation. SAHI is most valuable when you cannot retrain, such as when deploying a third-party model on a new domain."

---
# Domain Shift Analysis — Testing on Unseen Internet Images

**What is domain shift?**  
A model trained on one data distribution may fail on another distribution, even if the task looks similar. This is one of the most important concepts in production ML deployment.

**Our test:** Run both fine-tuned and generic YOLOv8 on aerial images from the internet — images the model has never seen, from different contexts than training data.

In [ ]:
# Load both models for comparison
fine_model = YOLO(MODEL_PATH)      # your fine-tuned model
generic_model = YOLO("yolov8n.pt") # generic pretrained model

print("Fine-tuned model:", MODEL_PATH)
print("Generic model: yolov8n.pt (COCO pretrained)")

In [ ]:
# ── Test on internet images ──────────────────────────────────────────────────
# Place test images in results/test_samples/ folder
# Two test images used in this project:
#   gettyimages-1040003524-2048x2048.jpg  — aerial city square, crowd of people
#   istockphoto-1893843900-1024x1024.jpg  — aerial parking lot, cars

test_folder = "results/test_samples"
os.makedirs(test_folder, exist_ok=True)

images = (glob.glob(os.path.join(test_folder, "*.jpg")) +
          glob.glob(os.path.join(test_folder, "*.png")))

print(f"Found {len(images)} test images in {test_folder}")

if len(images) == 0:
    print("\nNo images found. Add aerial images to results/test_samples/ and rerun.")
    print("Good test images: aerial city squares, parking lots, crowds from above.")

In [ ]:
# Run comparison for each test image
domain_shift_results = []

for img_path in images:
    name = os.path.basename(img_path)
    
    fine_results   = fine_model(img_path, conf=0.25)
    gen_results    = generic_model(img_path, conf=0.25)
    
    fine_count = len(fine_results[0].boxes)
    gen_count  = len(gen_results[0].boxes)
    
    # Side-by-side plot
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    axes[0].imshow(gen_results[0].plot()[:, :, ::-1])
    axes[0].set_title(f"Generic YOLOv8\n{gen_count} detections", fontsize=13)
    axes[0].axis('off')
    
    axes[1].imshow(fine_results[0].plot()[:, :, ::-1])
    axes[1].set_title(f"Fine-tuned best.pt\n{fine_count} detections", fontsize=13)
    axes[1].axis('off')
    
    plt.suptitle(f"Domain Shift Test: {name}", fontsize=15, fontweight='bold')
    plt.tight_layout()
    
    save_path = f"results/internet_vs_{name}"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n{name}:")
    print(f"  Generic model:    {gen_count} detections")
    print(f"  Fine-tuned model: {fine_count} detections")
    
    domain_shift_results.append({
        "image": name, "generic": gen_count, "finetuned": fine_count
    })

## Domain Shift Analysis — Observed Results

| Image | Generic YOLOv8 | Fine-tuned best.pt | Analysis |
|-------|---------------|-------------------|----------|
| City square (Getty) | 18 | 11 | Fine-tuned more precise — ignored fountain (generic called it "boat"). Higher confidence on true persons. |
| Parking lot (iStock) | 81 | 0 | Complete domain shift failure — model never saw close-up cars during training |

## Why This Happens — Technical Explanation

**Training data characteristics:**
- High altitude (~50-200m), ~90° top-down view
- Objects are 4-8px relative to full image
- Forest/road/rural backgrounds
- Consistent lighting and drone camera characteristics

**Internet images:**
- Lower altitude, wider angle, closer zoom
- Cars are large and detailed — NOT tiny aerial dots
- Urban backgrounds with different textures

The model learned "aerial high-altitude tiny object" patterns. Close-up cars fall outside that learned distribution entirely.

## Production Fix

```python
# Route by camera altitude metadata
if camera_altitude_meters > 50:
    detections = fine_tuned_model(image)   # specialized
else:
    detections = generic_model(image)       # general fallback

# Or train on diverse altitudes to reduce domain dependency
```

## The Interview Story

> "Testing on real-world internet images revealed classic domain shift. My fine-tuned model achieved 92% mAP on aerial drone data but detected 0 cars in a parking lot that the generic model found 81 in — because the parking lot was shot at lower altitude and cars appeared large rather than tiny. This is not a failure, it's a fundamental ML concept: specialization and generalization trade off directly. In production I would solve this by routing inference based on camera altitude metadata, training on diverse altitude ranges, or maintaining a two-model ensemble. This experience directly informed how I think about model deployment in security-sensitive contexts."

---
# RL vs Supervised Learning — Connecting to Thesis Research

This project uses **supervised learning**. My MASc thesis applies **reinforcement learning** to DDoS detection in 5G networks. Here is the key difference:

| | This Project (SL) | Thesis Research (RL) |
|-|-------------------|-----------------------|
| **Signal** | Labeled bounding box annotations | Reward from environment (blocked attack = +1) |
| **Data** | 4,821 annotated images | Network traffic episodes |
| **Learning** | Minimize loss vs ground truth | Maximize cumulative reward |
| **Exploration** | None — fixed dataset | Yes — agent must try actions |
| **Question** | Where is the object? | What action should I take now? |

**Why RL doesn't work for object detection:**  
Detection requires knowing exact spatial coordinates — which requires labeled ground truth. RL has no mechanism to learn (x, y, width, height) without explicit spatial reward signals, which would require labels anyway.

**Where they combine:**  
RL for drone navigation + SL for object detection = autonomous aerial surveillance system.